preprocessing for the dataset

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/train-test.csv", parse_dates=["date"])
val = pd.read_csv("../data/validation.csv", parse_dates=["date"])
dec = pd.read_csv("../data/december-chart-inputs.csv", parse_dates=["date"])

df["rate_per_mile"] = df["posted_rate"] / df["distance"]

In [3]:
clean = df[(df["rate_per_mile"] > 1.4) & (df["rate_per_mile"] < 4)].copy()
clean["weight"] = clean["weight"].abs()
val["weight"] = val["weight"].abs()
print(len(df), len(clean))

48000 47323


removed 677 loads with wrong prices (1.4%), fixed negative weights with abs in train and validation

In [4]:
cities = df[["pickup", "pickup_lat", "pickup_lon"]].drop_duplicates()
cities = cities.rename(columns={"pickup": "city", "pickup_lat": "lat", "pickup_lon": "lon"})

dec = dec.merge(cities.rename(columns={"city": "pickup", "lat": "pickup_lat", "lon": "pickup_lon"}), on="pickup")
dec = dec.merge(cities.rename(columns={"city": "delivery", "lat": "delivery_lat", "lon": "delivery_lon"}), on="delivery")

print(len(dec))
dec[["pickup", "pickup_lat", "pickup_lon", "delivery", "delivery_lat", "delivery_lon"]].head(2)

31


,pickup,pickup_lat,pickup_lon,delivery,delivery_lat,delivery_lon
0,Lexington,36.99152,-84.99876,Fort Wayne,41.31561,-85.36206
1,Lexington,36.99152,-84.99876,Fort Wayne,41.31561,-85.36206


added lat/lon to December file from the city table (31 rows, no missing coordinates)

In [5]:
def add_features(data):
    data["month"] = data["date"].dt.month
    data["day_of_week"] = data["date"].dt.dayofweek
    data["equipment_code"] = data["equipment"].map({"Dry Van": 0, "Flatbed": 1, "Reefer": 2})
    return data

clean = add_features(clean)
val = add_features(val)
dec = add_features(dec)

print(clean[["date", "month", "day_of_week", "equipment", "equipment_code"]].head(3))
print(val["equipment_code"].isna().sum(), dec["equipment_code"].isna().sum())

        date  month  day_of_week equipment  equipment_code
0 2025-01-01      1            2   Dry Van               0
1 2025-01-01      1            2    Reefer               2
2 2025-01-01      1            2   Dry Van               0
0 0


added month, day_of_week and equipment_code to all 3 files (no unknown equipment)

In [6]:
import os
os.makedirs("../data/processed", exist_ok=True)

clean.to_csv("../data/processed/train_clean.csv", index=False)
val.to_csv("../data/processed/validation_clean.csv", index=False)
dec.to_csv("../data/processed/december_clean.csv", index=False)

saved the 3 processed files to data/processed for the model notebook